# 01 - Scaling Laws

## The core observation

Training loss on held-out text falls predictably as you increase model size,
dataset size, and compute. Kaplan et al. (2020) showed the relationship is a
power law in each axis. Hoffmann et al. (2022), the Chinchilla paper, refined
it: the compute-optimal frontier requires roughly equal token and parameter
scaling. For a fixed compute budget `C` (FLOPs), the optimal model trains
approximately `N* = sqrt(C / 120)` parameters on `D* = 20 * N*` tokens
under the simplifying assumptions `D/N = 20` and `C = 6ND`. This is a
planning approximation, not a universal fitted law for every architecture.

```text
Compute budget C  (FLOPs, from training run)
Optimal params  N* = sqrt(C / 120)   # ≈ 0.091 * sqrt(C)
Optimal tokens  D* = 20 * N*
```

A quick sanity check: 6 * N* * D* = 6 * N* * 20 * N* = 120 * N*^2 = 120 * (C/120) = C. The
compute budget is fully used. GPT-3 (175B params, ~300B tokens) trained parameter-heavy
relative to tokens. Chinchilla (70B params, 1.4T tokens) used a 20:1 token-to-parameter
ratio. It outperformed the larger 280B-parameter Gopher across the reported
evaluation suite at a comparable training-compute budget.

## What the law measures and what it does not

The empirical fits describe held-out language-model loss over the runs studied.
They do not directly predict:

- performance on a specific downstream task;
- emergent capabilities that appear discontinuously at scale;
- the effect of data quality, domain mix, or deduplication.

Data and architecture changes can alter the fit. Some deployment-focused recipes
train longer to reduce inference cost; 20 tokens per parameter is not a hard
minimum or maximum.

## Flop arithmetic

A transformer forward pass through one parameter requires roughly 2 FLOPs.
Training adds the backward pass: approximately 6 FLOPs per parameter per token.

```text
Training FLOPs ≈ 6 * N * D
```

This rough estimate omits details such as attention's sequence-length cost,
optimizer arithmetic, and recomputation. Measure compute or use a fuller cost
model when those terms matter.

## Why this matters for architecture choice

If data is abundant and compute is fixed, Chinchilla scaling favors smaller
models trained longer. Llama 3 8B was trained on 15T tokens, roughly 94x
the 20:1 allocation for its parameter count (15T / 160B is about 94), illustrating
why the training-only allocation need not be the deployment choice.

## Exit check

Given a training compute budget of 1e23 FLOPs, derive the Chinchilla-optimal
parameter count and token count. Then explain why a production team might
choose to train half as many parameters on twice as many tokens instead.

## Build an equal-budget experiment

Finish `budget_tokens(C, N) = floor(C/(6N))`. For each candidate N, compute
its own D; copying one D across models changes the compute budget. Check:
`6ND <= C < 6N(D+1)`. The small rounding remainder is less than one token's
estimated compute. This invariant is what the grader checks.

The experiment notebook trains tiny causal language models at multiple widths
and seeds, with the same token stream, held-out evaluation, and estimated compute
cap. It reports the actual parameter counts, tokens processed, budget utilization,
and validation loss. Learning-rate schedules are matched to each run's duration.
Predict the winner, run it, and explain disagreements with your prediction.

A tiny corpus and short runs do not reproduce the Chinchilla scaling law. Look
for confounders: repeated data, token rounding, optimizer settings, model shape,
validation variance, and uncounted attention work. The correct conclusion may be
that this experiment cannot identify a stable optimum.

Primary source: [Hoffmann et al., Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556).


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""Level 66 — Scaling Laws

Implement Chinchilla compute-optimal scaling and FLOP estimation.
"""
import math

def chinchilla_optimal(compute_flops: float) -> dict:
    """
    Given a training compute budget in FLOPs, return a dict with:
      'params'  - compute-optimal parameter count (N*)
      'tokens'  - compute-optimal token count (D*)
    
    Use: N* = sqrt(C / 120), D* = 20 * N*
    """
    raise NotImplementedError

def training_flops(params: int, tokens: int) -> float:
    """Return estimated training FLOPs = 6 * params * tokens."""
    raise NotImplementedError

def inference_memory_gb(params: int, bytes_per_param: int = 2) -> float:
    """Return model weight memory in GB. 1 GB = 1e9 bytes."""
    raise NotImplementedError

def budget_tokens(compute_flops: float, params: int) -> int:
    """Largest integer D satisfying 6*params*D <= compute_flops.
    Raise ValueError if params<=0 or compute_flops<0. Zero budget gives 0.
    This lets different model sizes share one compute budget.
    """
    raise NotImplementedError


In [ ]:
# auto-grader appended to learner code
import math

budget = 1e23
result = chinchilla_optimal(budget)
assert isinstance(result, dict), "return a dict"
N = result["params"]
D = result["tokens"]
expected_N = int(math.sqrt(budget / 120))
assert abs(N - expected_N) <= 1, f"params: expected ~{expected_N}, got {N}"
assert D == 20 * N, f"tokens should be 20*N, got {D}"

flops = training_flops(1_000_000, 10_000)
assert abs(flops - 6e10) < 1, f"flops wrong: {flops}"

mem = inference_memory_gb(7_000_000_000)
assert abs(mem - 14.0) < 0.1, f"memory: expected ~14GB, got {mem:.2f}GB"

# Chinchilla-optimal for 2e23
r2 = chinchilla_optimal(2e23)
assert r2["tokens"] == 20 * r2["params"], "D* = 20*N* must hold"
for _c in [0, 1000, 12000, 1e8]:
    for _n in [1, 7, 1000]:
        _d = budget_tokens(_c, _n)
        assert isinstance(_d, int)
        assert 6 * _n * _d <= _c < 6 * _n * (_d + 1), 'maximal allocation inside budget'
for _c, _n in [(-1, 2), (100, 0), (100, -1)]:
    try:
        budget_tokens(_c, _n)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid allocation must raise ValueError')

print(f"N* = {result['params']:,}")
print(f"D* = {result['tokens']:,}")
print(f"Training FLOPs estimate: {training_flops(r2['params'], r2['tokens']):.2e}")
print("\n+150 XP — Scaling Laws complete.")


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import math

def chinchilla_optimal(compute_flops):
    n = int(math.sqrt(compute_flops / 120))
    return {'params': n, 'tokens': 20 * n}

def training_flops(params, tokens):
    return 6 * params * tokens

def inference_memory_gb(params, bytes_per_param=2):
    return params * bytes_per_param / 1e9

def budget_tokens(compute_flops, params):
    if params <= 0 or compute_flops < 0:
        raise ValueError('invalid budget or parameter count')
    return int(compute_flops // (6 * params))


In [ ]:
# auto-grader appended to learner code
import math

budget = 1e23
result = chinchilla_optimal(budget)
assert isinstance(result, dict), "return a dict"
N = result["params"]
D = result["tokens"]
expected_N = int(math.sqrt(budget / 120))
assert abs(N - expected_N) <= 1, f"params: expected ~{expected_N}, got {N}"
assert D == 20 * N, f"tokens should be 20*N, got {D}"

flops = training_flops(1_000_000, 10_000)
assert abs(flops - 6e10) < 1, f"flops wrong: {flops}"

mem = inference_memory_gb(7_000_000_000)
assert abs(mem - 14.0) < 0.1, f"memory: expected ~14GB, got {mem:.2f}GB"

# Chinchilla-optimal for 2e23
r2 = chinchilla_optimal(2e23)
assert r2["tokens"] == 20 * r2["params"], "D* = 20*N* must hold"
for _c in [0, 1000, 12000, 1e8]:
    for _n in [1, 7, 1000]:
        _d = budget_tokens(_c, _n)
        assert isinstance(_d, int)
        assert 6 * _n * _d <= _c < 6 * _n * (_d + 1), 'maximal allocation inside budget'
for _c, _n in [(-1, 2), (100, 0), (100, -1)]:
    try:
        budget_tokens(_c, _n)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid allocation must raise ValueError')

print(f"N* = {result['params']:,}")
print(f"D* = {result['tokens']:,}")
print(f"Training FLOPs estimate: {training_flops(r2['params'], r2['tokens']):.2e}")
print("\n+150 XP — Scaling Laws complete.")


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
"""Small, equal-estimated-compute causal-LM runs, not a Chinchilla reproduction.

CPU: uv run --with torch python experiments/scaling_runs.py
"""
import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)

class TinyLanguageModel(nn.Module):
    def __init__(self, width, vocab=16, context=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width)
        self.position = nn.Parameter(torch.zeros(context, width))
        self.norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.projection = nn.Linear(width, width)
        self.ffnorm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.head = nn.Linear(width, vocab)

    def forward(self, ids):
        x = self.embedding(ids) + self.position[:ids.shape[1]]
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=-1)
        x = x + self.projection(F.scaled_dot_product_attention(q, k, v, is_causal=True))
        return self.head(x + self.ff(self.ffnorm(x)))

def examples(count, seed):
    """Synthetic sequences: next symbol is the sum of the previous two mod 16."""
    generator = torch.Generator().manual_seed(seed)
    data = torch.zeros(count, 9, dtype=torch.long)
    data[:, :2] = torch.randint(0, 16, (count, 2), generator=generator)
    for i in range(2, 9):
        data[:, i] = (data[:, i-1] + data[:, i-2]) % 16
    return data[:, :-1], data[:, 1:]

def run_scaling(budget=3e7):
    train_x, train_y = examples(512, 101)
    val_x, val_y = examples(128, 202)
    rows = []
    for seed in [7, 17, 27]:
        for width in [8, 16, 32]:
            torch.manual_seed(seed)
            model = TinyLanguageModel(width)
            n = sum(p.numel() for p in model.parameters())
            tokens_per_step = 4 * 8
            steps = int(budget // (6 * n * tokens_per_step))
            assert steps > 0
            optimizer = torch.optim.AdamW(model.parameters(), lr=.01)
            for step in range(steps):
                # Match schedules to each run's duration and expose batch rounding.
                warmup = max(1, int(.1 * steps))
                warm = min(1, (step + 1) / warmup)
                cosine = .5 * (1 + math.cos(math.pi * step / max(1, steps)))
                for group in optimizer.param_groups:
                    group['lr'] = .01 * warm * cosine
                idx = (torch.arange(4) + step * 4) % len(train_x)
                optimizer.zero_grad()
                loss = F.cross_entropy(model(train_x[idx]).flatten(0, 1), train_y[idx].flatten())
                loss.backward(); optimizer.step()
            model.eval()
            with torch.no_grad():
                loss = F.cross_entropy(model(val_x).flatten(0, 1), val_y.flatten()).item()
            used = 6 * n * steps * tokens_per_step
            assert used <= budget < used + 6 * n * tokens_per_step
            rows.append(dict(seed=seed, width=width, params=n, tokens=steps*tokens_per_step,
                             steps=steps, estimated_flops=used, utilization=used/budget,
                             validation_loss=loss))
    return rows

if __name__ == '__main__':
    RESULTS = run_scaling()
    print(json.dumps(RESULTS, indent=2))
    print('Synthetic task; held-out draws can share patterns with training. FLOPs use 6ND, not hardware counters. No scaling exponent or universal optimum is established.')
